# BTL-1 Colab Training Notebook

This notebook is the release-check companion for BTL-1.

It should keep us aligned on the real ambition:
- open weights
- published dataset
- reproducible training
- local multi-tool execution on consumer hardware
- a clean handoff from data to adapter to eval to publish


In [ ]:
import subprocess
import sys
from pathlib import Path

def pip_install(*packages: str) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    subprocess.check_call(cmd)

pip_install(
    "datasets",
    "transformers",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "sentencepiece",
)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"drive mount skipped: {exc}")

print("Colab stack ready.")


## What Success Looks Like

By the end of the run, we want to be able to say:

- the dataset is clean and reproducible
- the training recipe is fixed, not hand-wavy
- the adapter exports cleanly
- the quantized model still follows tool-chain format
- the eval numbers can be reproduced by someone else without a tour guide


In [ ]:
from pathlib import Path

def locate_btl_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "BTL-1.md").exists():
            return candidate
        for child in candidate.iterdir():
            if child.is_dir() and (child / "BTL-1.md").exists():
                return child
    raise FileNotFoundError("Could not find BTL-1.md in or above the current workspace")

project = locate_btl_root()
data_dir = project / "data"

print(f"project root: {project}")
print("key artifacts:")
for rel in [
    "BTL-1.md",
    "data/raw/train.jsonl",
    "data/raw/eval.jsonl",
    "data/final/train.jsonl",
    "data/final/eval.jsonl",
    "data/templates",
    "data/generators/template-generator.mjs",
]:
    path = project / rel
    status = "ok" if path.exists() else "missing"
    print(f"- {rel}: {status}")


In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=str(project / "data" / "final" / "train.jsonl"), split="train")
eval_ds = load_dataset("json", data_files=str(project / "data" / "final" / "eval.jsonl"), split="train")

print(f"train rows: {len(train_ds)}")
print(f"eval rows: {len(eval_ds)}")
print(f"columns: {train_ds.column_names}")
print(train_ds[0]["messages"][0]["content"][:120])


## QLoRA Training

This is the part that turns the cleaned corpus into something we can actually ship.

The defaults below are tuned for a compact packed run on Colab or Lightning: 448-token windows, sequence packing, and a curated ~72k-row high-signal slice so the main loop stays quick enough to iterate without burning the whole session.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForLanguageModeling, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import Dataset
import json
import random
import re
from collections import defaultdict
import torch

base_model_name = "Qwen/Qwen2.5-7B-Instruct"
artifact_dir = project / "artifacts" / "qlora-adapter"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Set this to a small integer for a cheap smoke test. Leave it None for the full run.
smoke_rows = None
# Keep periodic eval small so the full run stays inside a sane budget.
eval_rows = 256
max_seq_length = 448
train_rows = 72000
packing = True
train_batch_size = 16
eval_batch_size = 16
grad_accum = 2
epochs = 1
learning_rate = 1e-4
dataloader_num_workers = 4

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
use_gradient_checkpointing = False
if use_gradient_checkpointing:
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def trim_rows(ds):
    if smoke_rows is None:
        return ds
    return ds.select(range(min(smoke_rows, len(ds))))

train_slice = trim_rows(train_ds)
eval_slice = trim_rows(eval_ds)
eval_monitor_slice = eval_slice.select(range(min(eval_rows, len(eval_slice))))

def render_messages(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def to_text(batch):
    return {"text": [render_messages(messages) for messages in batch["messages"]]}

train_text = train_slice.map(to_text, batched=True, remove_columns=train_slice.column_names)
eval_text = eval_monitor_slice.map(to_text, batched=True, remove_columns=eval_monitor_slice.column_names)

print(train_text[0]["text"][:240])


In [ ]:
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_seq_length, padding=False)

def pack_dataset(dataset, block_size):
    if not packing:
        return dataset

    packed_rows = []
    stash_ids = []
    stash_mask = []
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos_id

    for row in dataset:
        ids = list(row["input_ids"])
        mask = list(row["attention_mask"])
        if eos_id is not None:
            ids.append(eos_id)
            mask.append(1)
        stash_ids.extend(ids)
        stash_mask.extend(mask)
        while len(stash_ids) >= block_size:
            packed_rows.append({
                "input_ids": stash_ids[:block_size],
                "attention_mask": stash_mask[:block_size],
            })
            stash_ids = stash_ids[block_size:]
            stash_mask = stash_mask[block_size:]

    if stash_ids:
        if pad_id is None:
            pad_id = 0
        pad_len = block_size - len(stash_ids)
        packed_rows.append({
            "input_ids": stash_ids + [pad_id] * pad_len,
            "attention_mask": stash_mask + [0] * pad_len,
        })

    if not packed_rows:
        return Dataset.from_dict({"input_ids": [], "attention_mask": []})
    return Dataset.from_list(packed_rows)

train_tok = train_text.map(tokenize_batch, batched=True, remove_columns=train_text.column_names)
eval_tok = eval_text.map(tokenize_batch, batched=True, remove_columns=eval_text.column_names)
train_tok = pack_dataset(train_tok, max_seq_length)
eval_tok = pack_dataset(eval_tok, max_seq_length)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="/home/zeus/btl-1/checkpoints",
    num_train_epochs=epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=grad_accum,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=learning_rate,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=use_gradient_checkpointing,
    dataloader_num_workers=dataloader_num_workers,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
)

train_result = trainer.train()
trainer.save_state()
print(train_result)


## Adapter Export

This is the artifact we hand off after training: the LoRA adapter, tokenizer files, and a quick reload check so we do not ship a busted checkpoint.


In [ ]:
adapter_dir = artifact_dir / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

reloaded_base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    device_map="auto",
)
reloaded = PeftModel.from_pretrained(reloaded_base, adapter_dir)

print("adapter files:")
for item in sorted(adapter_dir.iterdir()):
    print("-", item.name)

print("reload ok:", type(reloaded).__name__)


## Release Gates

The notebook is not just a scratchpad. It is supposed to answer a few hard questions before we call a run shippable:

1. Does the raw corpus parse cleanly?
2. Does the final corpus stay distinct from raw?
3. Do the templates match the cleaned schema?
4. Can we summarize the exact training recipe without guessing?
5. Could another person rerun the same flow from a fresh checkout?


In [ ]:
import hashlib
import json
import re

allowed_tools = {
    "file_search", "read_file", "write_file", "email", "web_search",
    "browse", "clipboard", "shell_command", "notes", "reasoning",
}

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def audit_jsonl(path: Path) -> dict:
    lines = 0
    malformed = 0
    email_missing_body = 0
    file_search_name = 0
    write_ppt = 0
    unknown_tools = 0
    multi_tool = 0

    with path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            lines += 1
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                malformed += 1
                continue

            assistant = next((msg for msg in row.get("messages", []) if msg.get("role") == "assistant"), None)
            if not assistant:
                malformed += 1
                continue

            content_clean = re.sub(r'<reasoning>.*?</reasoning>\s*', '', assistant["content"], flags=re.DOTALL).strip()
            payload = json.loads(content_clean)
            if isinstance(payload, list) and len(payload) > 1:
                multi_tool += 1

            for step in payload:
                if step.get("tool") not in allowed_tools:
                    unknown_tools += 1
                params = step.get("params") or {}
                if step.get("tool") == "email" and "body" not in params:
                    email_missing_body += 1
                if step.get("tool") == "file_search" and "name" in params:
                    file_search_name += 1
                if step.get("tool") == "write_file" and isinstance(params.get("path"), str) and params["path"].endswith((".ppt", ".pptx")):
                    write_ppt += 1

    return {
        "lines": lines,
        "malformed": malformed,
        "unknown_tools": unknown_tools,
        "email_missing_body": email_missing_body,
        "file_search_name": file_search_name,
        "write_ppt": write_ppt,
        "multi_tool_pct": round((multi_tool / lines) * 100, 2) if lines else 0.0,
        "sha256": sha256(path)[:16],
    }

for label, rel in {
    "raw/train": "data/raw/train.jsonl",
    "raw/eval": "data/raw/eval.jsonl",
    "final/train": "data/final/train.jsonl",
    "final/eval": "data/final/eval.jsonl",
}.items():
    report = audit_jsonl(project / rel)
    print(f"{label:11} lines={report['lines']:7} multi={report['multi_tool_pct']:5.2f}% sha={report['sha256']}")
    print(f"  malformed={report['malformed']} unknown={report['unknown_tools']} email_missing_body={report['email_missing_body']} file_search_name={report['file_search_name']} write_ppt={report['write_ppt']}")


## Training Recipe

The notebook should mirror the actual plan from `BTL-1.md`:

- base model: Qwen 2.5 7B
- method: packed QLoRA on a curated ~72k-row subset
- LoRA shape: rank 64, alpha 128
- platform: Colab A100
- output: adapter first, then GGUF for local inference
- goal: a model that can keep the JSON tool-chain format intact under pressure

The important part is not the marketing line. It is the repeatable sequence: data check, train, export, quantize, smoke test, eval, publish.


In [ ]:
training_plan = {
    "base_model": "Qwen 2.5 7B",
    "method": "Packed QLoRA",
    "lora_rank": 64,
    "lora_alpha": 128,
    "max_seq_length": 448,
    "packing": True,
    "train_batch_size": 16,
    "eval_batch_size": 16,
    "grad_accum": 2,
    "eval_rows": 256,
    "train_rows": 72000,
    "dataloader_num_workers": 4,
    "epochs": 1,
    "platform": "Lightning A100 / Colab A100",
    "target_outputs": ["LoRA adapter", "Q4_K_M GGUF"],
    "success_metric": "tool-chain accuracy stays stable after export",
}

for key, value in training_plan.items():
    print(f"{key}: {value}")

print("\nrun order:")
print("1. validate data and templates")
print("2. train adapter")
print("3. export and quantize")
print("4. smoke test the local agent wrapper")
print("5. run tool-chain and reasoning evals")
print("6. package weights, data, recipe, and notes for publish")


## Handoff Standard

When this notebook gets used in anger, the handoff should include:

- what was changed
- where the artifacts live
- what validation ran
- what still needs human judgment

If the run is not ready to publish, say why in one sentence and name the next unblocker.


## Adapter Eval

Run the trained adapter against held-out eval examples. Measures:
- exact tool match (right tool in right order)
- param field presence (required params exist)
- JSON parsability (model outputs valid JSON)

This is the gate before GGUF export. If accuracy is below 80%, investigate before shipping.


In [ ]:
import json
import re

eval_samples = eval_slice.select(range(min(200, len(eval_slice))))
results = []

for i, sample in enumerate(eval_samples):
    messages = sample["messages"]
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    expected_raw = re.sub(r'<reasoning>.*?</reasoning>\s*', '', messages[-1]["content"], flags=re.DOTALL).strip()
    expected = json.loads(expected_raw)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
    decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        predicted_raw = re.sub(r'<reasoning>.*?</reasoning>\s*', '', decoded, flags=re.DOTALL).strip()
        predicted = json.loads(predicted_raw)
        is_valid_json = True
    except (json.JSONDecodeError, ValueError):
        predicted = []
        is_valid_json = False

    if not isinstance(predicted, list):
        predicted = [predicted] if isinstance(predicted, dict) else []
        is_valid_json = True

    correct_tools = len(expected) == len(predicted) and all(
        e["tool"] == p.get("tool") for e, p in zip(expected, predicted)
    )
    param_fields_ok = all(
        p.get("params") is not None and len(p["params"]) > 0
        for p in predicted
    ) if predicted else False

    results.append({
        "valid_json": is_valid_json,
        "correct_tools": correct_tools,
        "param_fields_ok": param_fields_ok,
        "expected": expected,
        "predicted": predicted,
        "raw_output": decoded[:240],
    })

json_ok = sum(1 for r in results if r["valid_json"])
tool_ok = sum(1 for r in results if r["correct_tools"])
param_ok = sum(1 for r in results if r["param_fields_ok"])
total = len(results)

print(f"eval samples: {total}")
print(f"valid JSON:   {json_ok}/{total} ({100*json_ok//total}%)")
print(f"tool match:   {tool_ok}/{total} ({100*tool_ok//total}%)")
print(f"params ok:    {param_ok}/{total} ({100*param_ok//total}%)")
print()
for r in results[:3]:
    print("EXP:", json.dumps(r["expected"]))
    print("GOT:", json.dumps(r["predicted"]))
    print()


## GGUF Export

Convert the merged model to Q4_K_M GGUF for local inference via llama.cpp.

Requires `llama.cpp` to be installed. We clone it briefly, build the quantize tool, and export.
The output GGUF goes to `artifacts/btl-1-q4_k_m.gguf`.


In [ ]:
import os
import subprocess
import shutil

gguf_dir = project / "artifacts" / "gguf"
gguf_dir.mkdir(parents=True, exist_ok=True)

# Step 1: merge LoRA into base model
merge_dir = project / "artifacts" / "merged"
merge_dir.mkdir(parents=True, exist_ok=True)

merged_model = reloaded.merge_and_unload()
merged_model.save_pretrained(merge_dir)
tokenizer.save_pretrained(merge_dir)
print("merged model saved to", merge_dir)

# Step 2: convert to FP16 GGUF using llama.cpp convert script
llama_cpp_dir = project / "artifacts" / "llama.cpp"
if not llama_cpp_dir.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(llama_cpp_dir)])

convert_py = llama_cpp_dir / "convert_hf_to_gguf.py"
fp16_gguf = gguf_dir / "btl-1-fp16.gguf"

subprocess.check_call([
    sys.executable, str(convert_py),
    str(merge_dir),
    "--outfile", str(fp16_gguf),
    "--outtype", "f16",
])
print("FP16 GGUF:", fp16_gguf)

# Step 3: quantize to Q4_K_M
q4_gguf = gguf_dir / "btl-1-q4_k_m.gguf"
quantize_bin = llama_cpp_dir / "build" / "bin" / "quantize"

if quantize_bin.exists():
    subprocess.check_call([str(quantize_bin), str(fp16_gguf), str(q4_gguf), "Q4_K_M"])
    print("Q4_K_M GGUF:", q4_gguf)
    print("size MB:", round(q4_gguf.stat().st_size / 1e6, 1))
else:
    print("quantize binary not found, building llama.cpp...")
    build_dir = llama_cpp_dir / "build"
    subprocess.check_call(["cmake", "-B", str(build_dir), "-S", str(llama_cpp_dir), "-DLLAMA_CUDA=ON"], cwd=str(llama_cpp_dir))
    subprocess.check_call(["cmake", "--build", str(build_dir), "--config", "Release", "--target", "quantize"], cwd=str(llama_cpp_dir))
    subprocess.check_call([str(quantize_bin), str(fp16_gguf), str(q4_gguf), "Q4_K_M"])
    print("Q4_K_M GGUF:", q4_gguf)
    print("size MB:", round(q4_gguf.stat().st_size / 1e6, 1))


## Final Benchmark Suite

Run the saved adapter against BTL-1 held-out eval and any external benchmark JSONL files we have mounted. Missing files are skipped, which keeps the notebook usable in Colab without extra setup.


In [ ]:
import sys
import os

sys.path.insert(0, str(project))
from eval.run_all import run_suite, render_suite

def generate_text(prompt: str, max_new_tokens: int = 512) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(reloaded.device)
    with torch.no_grad():
        outputs = reloaded.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.0, do_sample=False)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

suite_limit = int(os.environ.get("BTL_EVAL_LIMIT", "0")) or None
suite = run_suite(generate_text, project_root=project, tokenizer=tokenizer, limit=suite_limit)
print(render_suite(suite))
